# 02 Backtest Temperature Forecast

This notebook uses `skforecast` on the local weather series.
It stays intentionally small so the migration to AGILAB is easy to explain.


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster

csv_path = Path("../data/meteo_fr_daily_sample.csv")
df = pd.read_csv(csv_path, parse_dates=["date"]).sort_values("date")
series = df.set_index("date")["tmax_c"]


In [ ]:
forecaster = ForecasterRecursive(
    regressor=RandomForestRegressor(random_state=42, n_estimators=100),
    lags=7,
)

cv = TimeSeriesFold(steps=7, initial_train_size=24)
metric, predictions = backtesting_forecaster(
    forecaster=forecaster,
    y=series,
    cv=cv,
    metric="mean_absolute_error",
    n_jobs=1,
    verbose=False,
)
metric, predictions.head()


## Migration note

In AGILAB this becomes two explicit stages:

- `build_features`
- `backtest_forecaster`

The important change is that metrics and predictions are exported as stable artifacts instead of remaining attached to the notebook kernel.
